# 05 Workforce

Converts CCC (step 03's output) to an employee estimate, splits off resort-provided beds, and reports the remaining workforce that would need housing elsewhere. The one evidence-backed employees-per-CCC ratio available is a single comparator resort (Sunshine Village), not RMR itself and not a low/mid/high range from any source, so this notebook is explicit about which numbers are evidence and which are a judgment-call sensitivity band built around that one data point.

In [ ]:
research_dir = "research"
processed_dir = "data/processed"
sensitivity_band = (0.7, 1.0, 1.3)  # low, mid, high multipliers on the one real ratio

## Load the ledger and step 03's CCC output

Confirms C040 (the one real capacity-to-employee data point) still points to its source, and reads Buildout CCC from step 03 rather than retyping it.

In [ ]:
import sys

import pandas as pd

sys.path.insert(0, "src")
from resort.ledger import read_ledger

ledger = read_ledger(
    claims_path=f"{research_dir}/claims.csv",
    sources_path=f"{research_dir}/sources.csv",
)
assert ledger["claims"]["C040"]["source_id"] == "S038"

ccc_by_phase = pd.read_csv(f"{processed_dir}/02_ccc_by_phase.csv")
ccc_by_phase

## The one real capacity-to-employee ratio (claim C040)

Parks Canada's Sunshine Village site guidelines state, on the same page, both a 6,000-skier capacity and roughly 700 peak-season employees for that resort. This is a comparator, not RMR, and a single data point, not a range; research-scout could not find a second source giving both halves of the relationship for any resort. The resulting ratio is kept as one number, labelled by its source, not blended with any other guess.

In [ ]:
sunshine_village_ccc = 6000
sunshine_village_peak_employees = 700
employees_per_ccc_skier = sunshine_village_peak_employees / sunshine_village_ccc
employees_per_ccc_skier

## Apply the ratio to RMR's CCC, with a sensitivity band (judgment call)

The ratio itself is evidence (C040). The low/mid/high multipliers applied to it are not: no source gives RMR-specific or industry-wide low/mid/high staffing ratios, so this notebook picks a +/-30% band around the one real ratio as a transparent sensitivity range, not as three separately evidenced scenarios. This is recorded plainly rather than presented as if three distinct ratios were each independently sourced.

In [ ]:
low_mult, mid_mult, high_mult = sensitivity_band
employee_estimates = ccc_by_phase.copy()
for label, mult in [("low", low_mult), ("mid", mid_mult), ("high", high_mult)]:
    employee_estimates[f"employees_{label}"] = (
        employee_estimates["ccc_skiers"] * employees_per_ccc_skier * mult
    ).round().astype(int)
employee_estimates

## Subtract resort-provided beds

Reads step 02's employee-housing output (C031's 3 x 150-200 beds) rather than retyping it. C011/C012 (site capacity over 400, and a 2022 plan of 3 x 160 = 480) sit inside this range; source-verifier separately flagged that the building actually opened in 2025 (S020) may differ in unit mix from the 2022 plan (C012's notes), so this is treated as a range, not a single figure, and that discrepancy is left for steps/05's Open issues rather than resolved here.

In [ ]:
import json

with open(f"{processed_dir}/02_employee_housing_phase2.json", encoding="utf-8") as f:
    housing = json.load(f)

for label in ("low", "mid", "high"):
    employee_estimates[f"needing_housing_{label}_min_beds"] = (
        employee_estimates[f"employees_{label}"] - housing["total_beds_max"]
    ).clip(lower=0)
    employee_estimates[f"needing_housing_{label}_max_beds"] = (
        employee_estimates[f"employees_{label}"] - housing["total_beds_min"]
    ).clip(lower=0)
employee_estimates[
    ["phase", "employees_low", "employees_mid", "employees_high",
     "needing_housing_mid_min_beds", "needing_housing_mid_max_beds"]
]

## Write outputs

The full table (all phases, all three sensitivity bands, both housing-subtraction bounds), so step 08's synthesis can pick the phase and band it needs without recomputing anything.

In [ ]:
import os

os.makedirs(processed_dir, exist_ok=True)
employee_estimates.to_csv(f"{processed_dir}/05_employee_estimates.csv", index=False, encoding="utf-8")
print("wrote 05_employee_estimates.csv")

## Checks

Employee counts must rise with CCC (more capacity implies more staff under a constant ratio), low must be below mid below high at every phase, and beds-needed must never go negative (already enforced by clip(lower=0) above, checked again here explicitly).

In [ ]:
assert (employee_estimates["employees_low"] <= employee_estimates["employees_mid"]).all()
assert (employee_estimates["employees_mid"] <= employee_estimates["employees_high"]).all()
assert employee_estimates["employees_mid"].is_monotonic_increasing, "employees should rise with CCC across phases"
assert (employee_estimates.filter(like="needing_housing") >= 0).all().all()
print("checks passed")

## Versions

In [ ]:
import importlib.metadata
import sys

print("python", sys.version)
for pkg in ["pandas"]:
    print(pkg, importlib.metadata.version(pkg))